# Lab 6: Interactive Visual Analytics — Folium Map

**Goal:** Build an interactive map showing all SpaceX launch sites, color-coded launch outcomes at each site, and a proximity analysis (distance from a launch site to the nearest coastline, city, railway, or highway) to explore whether geography matters for launch site selection.

In [1]:
!pip install folium -q

In [2]:
import pandas as pd
import folium
from folium.plugins import MarkerCluster, MousePosition
from folium.features import DivIcon
from math import sin, cos, sqrt, atan2, radians
import os

## 0. Load the dataset

In [3]:
if os.path.exists('dataset_part_2.csv'):
    df = pd.read_csv('dataset_part_2.csv')
else:
    print('dataset_part_2.csv not found locally, loading from IBM dataset repository...')
    fallback_url = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_2.csv'
    df = pd.read_csv(fallback_url)

launch_sites_df = df.groupby('LaunchSite', as_index=False).first()[['LaunchSite', 'Latitude', 'Longitude']]
launch_sites_df

dataset_part_2.csv not found locally, loading from IBM dataset repository...


,LaunchSite,Latitude,Longitude
0,CCAFS SLC 40,28.561857,-80.577366
1,KSC LC 39A,28.608058,-80.603956
2,VAFB SLC 4E,34.632093,-120.610829


## 1. Base map with a marker for every launch site

In [4]:
site_map = folium.Map(location=[28.5, -80.5], zoom_start=4.5)

for _, row in launch_sites_df.iterrows():
    coordinate = [row['Latitude'], row['Longitude']]
    circle = folium.Circle(coordinate, radius=1000, color='#d35400', fill=True).add_child(
        folium.Popup(row['LaunchSite'])
    )
    marker = folium.map.Marker(
        coordinate,
        icon=DivIcon(
            icon_size=(20, 20),
            icon_anchor=(0, 0),
            html=f'<div style="font-size: 12px; color:#d35400;"><b>{row["LaunchSite"]}</b></div>'
        )
    )
    site_map.add_child(circle)
    site_map.add_child(marker)

site_map

## 2. Mark every individual launch, colored green (success) / red (failure)

In [5]:
marker_cluster = MarkerCluster()
df['marker_color'] = df['Class'].apply(lambda x: 'green' if x == 1 else 'red')

site_map2 = folium.Map(location=[28.5, -80.5], zoom_start=4.5)
site_map2.add_child(marker_cluster)

for _, record in df.iterrows():
    marker = folium.Marker(
        [record['Latitude'], record['Longitude']],
        icon=folium.Icon(color='white', icon_color=record['marker_color']),
        popup=f"{record['LaunchSite']} — {'Success' if record['Class']==1 else 'Failure'}"
    )
    marker_cluster.add_child(marker)

site_map2

## 3. Proximity analysis: distance from a launch site to the coastline
We use the Haversine formula. Coordinates below are for the coastline point nearest **KSC LC-39A** — if your report focuses on a different site, replace `coastline_lat/lon` with the nearest coastline point you find on the map for that site (right-click on the map in Colab, use MousePosition to read coordinates).

In [6]:
def calculate_distance(lat1, lon1, lat2, lon2):
    R = 6373.0  # Earth's radius in km
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    return R * c

# See the exact site names available in your data:
print(launch_sites_df['LaunchSite'].tolist())

# Pick a site by partial match instead of an exact hardcoded string, so this
# works regardless of exact spacing/formatting in your dataset (e.g. 'KSC LC-39A',
# 'KSC LC 39A', 'CCAFS LC-40', etc). Change the search text below if needed.
site_search = 'KSC'
matches = launch_sites_df[launch_sites_df['LaunchSite'].str.contains(site_search, case=False)]

if matches.empty:
    # fall back to the first available site if the search text doesn't match anything
    chosen_site = launch_sites_df.iloc[0]
else:
    chosen_site = matches.iloc[0]

site_name = chosen_site['LaunchSite']
launch_site_lat = chosen_site['Latitude']
launch_site_lon = chosen_site['Longitude']
print(f"Using site: {site_name} ({launch_site_lat}, {launch_site_lon})")

# Nearest coastline point (approximate, read from map -- adjust if you picked a different site)
coastline_lat = 28.56367
coastline_lon = -80.57163

distance_coastline = calculate_distance(launch_site_lat, launch_site_lon, coastline_lat, coastline_lon)
print(f"Distance from {site_name} to nearest coastline: {distance_coastline:.2f} km")

['CCAFS SLC 40', 'KSC LC 39A', 'VAFB SLC 4E']
Using site: KSC LC 39A (28.6080585, -80.6039558)
Distance from KSC LC 39A to nearest coastline: 5.86 km


In [7]:
proximity_map = folium.Map(location=[launch_site_lat, launch_site_lon], zoom_start=12)

folium.Marker([launch_site_lat, launch_site_lon], popup=site_name,
              icon=folium.Icon(color='orange')).add_to(proximity_map)
folium.Marker([coastline_lat, coastline_lon], popup='Nearest coastline point',
              icon=folium.Icon(color='blue')).add_to(proximity_map)

folium.PolyLine(locations=[[launch_site_lat, launch_site_lon], [coastline_lat, coastline_lon]],
                 weight=2, color='blue').add_to(proximity_map)
folium.map.Marker(
    [(launch_site_lat + coastline_lat) / 2, (launch_site_lon + coastline_lon) / 2],
    icon=DivIcon(icon_size=(20, 20), icon_anchor=(0, 0),
                 html=f'<div style="font-size:12px;color:#252850;"><b>{distance_coastline:.2f} KM</b></div>')
).add_to(proximity_map)

proximity_map

## 4. Save maps as standalone HTML (for your GitHub repo / slides)

In [8]:
site_map.save('launch_sites_map.html')
site_map2.save('launch_outcomes_map.html')
proximity_map.save('proximity_analysis_map.html')
print('Saved 3 HTML maps')

Saved 3 HTML maps
